In [0]:
from pyspark.sql import functions
from pyspark.sql.types import *
print("BANKING FRAUD MONITORING - DATA PROFILING")

In [0]:
dataset_base = "/Volumes/capstone_team01/bronze/raw_data/datasets/"
transaction_path = f"{dataset_base}/transaction_data/card_transactions/"
event_path = f"{dataset_base}/event_data/device_authentication_events.csv"
cdc_path = f"{dataset_base}/cdc_data/payment_status_changes.csv"

In [0]:
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("transaction_ts", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("country", StringType(), True),
    StructField("authorization_code", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("device_id", StringType(), True)
])

authentication_schema = StructType([
    StructField("auth_event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("authentication_method", StringType(), True),
    StructField("authentication_result", StringType(), True),
    StructField("ip_country", StringType(), True),
    StructField("attempt_number", IntegerType(), True),
    StructField("risk_signal", StringType(), True)
])

status_schema = StructType([
    StructField("change_event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("change_ts", StringType(), True),
    StructField("old_status", StringType(), True),
    StructField("new_status", StringType(), True),
    StructField("reason_code", StringType(), True),
    StructField("change_sequence", IntegerType(), True),
    StructField("source_system", StringType(), True),
    StructField("operator_id", StringType(), True),
    StructField("correlation_id", StringType(), True)
])

In [0]:
transactions_df = (
    spark.read
    .option("header", True)
    .schema(transaction_schema)
    .csv(transaction_path)
)

authentication_df = (
    spark.read
    .option("header", True)
    .schema(authentication_schema)
    .csv(event_path)
)

status_changes_df = (
    spark.read
    .option("header", True)
    .schema(status_schema)
    .csv(cdc_path)
)

print("All datasets loaded successfully")

In [0]:
print("CARD TRANSACTIONS")
transactions_df.printSchema()
display(transactions_df.limit(5))

print("DEVICE AUTHENTICATION EVENTS")
authentication_df.printSchema()
display(authentication_df.limit(5))

print("PAYMENT STATUS CHANGES")
status_changes_df.printSchema()
display(status_changes_df.limit(5))

In [0]:
print("Transaction rows:", transactions_df.count())
print("Authentication rows:", authentication_df.count())
print("Status change rows:", status_changes_df.count())

In [0]:
# ============================================================
# 1. DUPLICATE ROWS (exact copies across all columns)
# ============================================================

datasets = {
    "CARD TRANSACTIONS": transactions_df,
    "DEVICE AUTHENTICATION EVENTS": authentication_df,
    "PAYMENT STATUS CHANGES": status_changes_df
}

duplicate_rows = []

for name, df in datasets.items():
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows.append((name, total_rows, distinct_rows, total_rows - distinct_rows))

display(
    spark.createDataFrame(
        duplicate_rows,
        "dataset STRING, total_rows LONG, distinct_rows LONG, duplicate_rows LONG"
    )
)

In [0]:
# ============================================================
# 2. UNIQUE IDS (business keys)
# ============================================================

key_columns = {
    "CARD TRANSACTIONS": (transactions_df, "transaction_id"),
    "DEVICE AUTHENTICATION EVENTS": (authentication_df, "auth_event_id"),
    "PAYMENT STATUS CHANGES": (status_changes_df, "change_event_id")
}

key_summary = None

for name, (df, key) in key_columns.items():
    summary = df.select(
        functions.lit(name).alias("dataset"),
        functions.lit(key).alias("key_column"),
        functions.count("*").alias("total_rows"),
        functions.countDistinct(key).alias("unique_ids"),
        functions.count(functions.when(functions.col(key).isNull(), 1)).alias("null_ids")
    ).withColumn(
        "repeated_id_rows",
        functions.col("total_rows") - functions.col("unique_ids") - functions.col("null_ids")
    )
    key_summary = summary if key_summary is None else key_summary.unionByName(summary)

display(key_summary)

# Transaction IDs that appear more than once
print("TRANSACTION IDS APPEARING MORE THAN ONCE")
display(
    transactions_df.groupBy("transaction_id")
    .count()
    .filter(functions.col("count") > 1)
    .orderBy(functions.col("count").desc(), "transaction_id")
)

In [0]:
# ============================================================
# 3. NULL COUNTS (per column, one row per dataset + column)
# ============================================================

null_rows = []

for name, df in datasets.items():
    total_rows = df.count()
    counts = df.select([
        functions.count(functions.when(functions.col(c).isNull(), 1)).alias(c)
        for c in df.columns
    ]).first().asDict()
    for column, nulls in counts.items():
        null_rows.append((name, column, nulls, round(100.0 * nulls / total_rows, 2)))

null_summary = spark.createDataFrame(
    null_rows,
    "dataset STRING, column_name STRING, null_count LONG, null_pct DOUBLE"
)

print("COLUMNS WITH NULLS")
display(null_summary.filter("null_count > 0").orderBy(functions.col("null_count").desc()))

print("ALL COLUMNS")
display(null_summary)

In [0]:
# ============================================================
# 4. CATEGORICAL DISTRIBUTIONS
# ============================================================

def category_distribution(df, column):

    print("\nCATEGORY DISTRIBUTION:", column)

    display(
        df.groupBy(column)
        .count()
        .orderBy(functions.col("count").desc())
    )


category_distribution(transactions_df, "payment_status")
category_distribution(transactions_df, "channel")
category_distribution(transactions_df, "currency")

category_distribution(authentication_df, "authentication_method")
category_distribution(authentication_df, "authentication_result")

category_distribution(status_changes_df, "old_status")
category_distribution(status_changes_df, "new_status")

In [0]:
# ============================================================
# 5. NUMERIC ANALYSIS
# ============================================================

display(
    transactions_df.select(
        functions.min("amount").alias("minimum_amount"),
        functions.max("amount").alias("maximum_amount"),
        functions.avg("amount").alias("average_amount"),
        functions.expr(
            "percentile_approx(amount, 0.5)"
        ).alias("median_amount"),
        functions.stddev("amount").alias("standard_deviation")
    )
)

display(
    authentication_df.select(
        functions.min("attempt_number").alias("minimum_attempt"),
        functions.max("attempt_number").alias("maximum_attempt"),
        functions.avg("attempt_number").alias("average_attempt")
    )
)

In [0]:
# ============================================================
# 6. TIMESTAMP RANGES
# ============================================================

display(
    transactions_df.select(
        functions.min("transaction_ts").alias("earliest_transaction"),
        functions.max("transaction_ts").alias("latest_transaction")
    )
)

display(
    authentication_df.select(
        functions.min("event_ts").alias("earliest_event"),
        functions.max("event_ts").alias("latest_event")
    )
)

display(
    status_changes_df.select(
        functions.min("change_ts").alias("earliest_change"),
        functions.max("change_ts").alias("latest_change")
    )
)

In [0]:
# ============================================================
# 7. DATA QUALITY CHECKS (rule violations)
# ============================================================

def count_where(df, condition):
    return df.filter(condition).count()

quality_checks = [
    ("CARD TRANSACTIONS", "amount <= 0",
        count_where(transactions_df, "amount <= 0")),
    ("CARD TRANSACTIONS", "amount >= 999999 (possible sentinel)",
        count_where(transactions_df, "amount >= 999999")),
    ("CARD TRANSACTIONS", "amount not numeric / null",
        count_where(transactions_df, "amount IS NULL")),
    ("CARD TRANSACTIONS", "transaction_ts not a valid timestamp",
        count_where(transactions_df, "transaction_ts IS NOT NULL AND try_cast(transaction_ts AS TIMESTAMP) IS NULL")),
    ("DEVICE AUTHENTICATION EVENTS", "attempt_number <= 0",
        count_where(authentication_df, "attempt_number <= 0")),
    ("DEVICE AUTHENTICATION EVENTS", "event_ts not a valid timestamp",
        count_where(authentication_df, "event_ts IS NOT NULL AND try_cast(event_ts AS TIMESTAMP) IS NULL")),
    ("PAYMENT STATUS CHANGES", "old_status = new_status (no-op change)",
        count_where(status_changes_df, "old_status = new_status")),
    ("PAYMENT STATUS CHANGES", "change_ts not a valid timestamp",
        count_where(status_changes_df, "change_ts IS NOT NULL AND try_cast(change_ts AS TIMESTAMP) IS NULL"))
]

display(
    spark.createDataFrame(
        quality_checks,
        "dataset STRING, check_name STRING, failing_rows LONG"
    )
)

In [0]:
# ============================================================
# 8. FOREIGN KEY CHECKS (events must reference a transaction)
# ============================================================

authentication_orphans = authentication_df.join(
    transactions_df.select("transaction_id").distinct(),
    "transaction_id",
    "left_anti"
)

print("AUTHENTICATION EVENTS WITHOUT TRANSACTIONS:", authentication_orphans.count())
display(authentication_orphans)


status_orphans = status_changes_df.join(
    transactions_df.select("transaction_id").distinct(),
    "transaction_id",
    "left_anti"
)

print("STATUS CHANGES WITHOUT TRANSACTIONS:", status_orphans.count())
display(status_orphans)

In [0]:
print("======================================")
print("BANKING FRAUD MONITORING")
print("DATA PROFILING COMPLETED")
print("======================================")

print("Transaction rows:", transactions_df.count())
print("Authentication rows:", authentication_df.count())
print("Status change rows:", status_changes_df.count())